# Dynamic Autoscaling: How Many Hosts Does SURF Need?

*Use case 1 of 3, about 10 minutes.*

We twin SURF-SARA, a production cluster at SURF, the Dutch national supercomputing center.
SURF-SARA contains 277 hosts (i.e., servers), each with 128 GB of RAM and 16 processing cores running at maximum 2.1 GHz.
OpenDT mimics (twins) the operation of SURF-SARA: it replays the jobs of one week of October 2022 through the OpenDC
simulator, and predicts the power draw of the cluster.
Because the digital twin is a simulation, we can explore changes without touching the physical datacenter (i.e., what-if
analysis).

In this notebook, we (1) validate OpenDT against the power draw measured at SURF, (2) explore what-if scenarios with
fewer and more hosts, and (3) identify how many hosts SURF needs over the day.

Run the cells in order (Shift+Enter). When short on time, skip the parts marked *optional*.

In [ ]:
import pandas as pd
from opendt_tutorial import *

check_setup()

## 1. Validating OpenDT against SURF

We simulate the first day of the SURF workload (6-7 October 2022, 787 jobs) and compare the predicted power draw with
the ground truth (grey), i.e., the power draw measured at SURF.
We quantify the error with the Mean Absolute Percentage Error (MAPE); the lower the MAPE, the more accurate the
prediction.

In [ ]:
DAY = TRACE_START + pd.Timedelta("24h")  # the first 24 hours of the trace

day1 = simulate(until=DAY, label="OpenDT")
plot_power(day1)
print(f"MAPE: {mape(day1):.1f}%")

*Question:* the community-accepted error rate is 10%. Is OpenDT accurate enough for what-if analysis? When is OpenDT less accurate?

<details><summary>Answer</summary>

Yes, the MAPE is about 6%, within 10%.
However, OpenDT predicts less than the ground truth in almost every minute of the day (under-estimation), more so in the
first 6 hours (MAPE 10.4%) than in the rest of the day (MAPE 4.6%).
The OpenDT paper identifies the same under-estimation trend. Notebook 2 explores how calibration reduces the error.
</details>

## 2. What-If Analysis: Fewer and More Hosts

Should SURF buy more hosts, or switch some off? In the physical datacenter, both are costly to try (e.g., buying
hardware, or slowing down the jobs of users).
With OpenDT, we simulate the same day with 1/4, 1/2, 1x, and 10x the hosts (69, 138, 277, and 2,770 hosts).

*Question:* with 10x the hosts, what power draw do we expect, compared to 277 hosts? Write down a guess, then run the
next cell.

In [ ]:
sizes = [69, 138, 277, 2770]
runs = simulate_many("hosts", sizes, until=DAY)

table = pd.DataFrame.from_dict({hosts: run.summary() for hosts, run in runs.items()}, orient="index")
table[["avg_power_kw", "jobs_waited", "mean_wait_min", "waiting_at_end"]].round(1)

*Columns:* `avg_power_kw` is the average power draw [kW]; `jobs_waited` counts the jobs that cannot start right
away, because all hosts are busy; `mean_wait_min` is the average waiting time per job [minutes]; `waiting_at_end`
counts the jobs still waiting at the end of the day.

*Question:* why do 10x the hosts not draw 10x the power?

<details><summary>Answer</summary>

The jobs are the same, so the 2,493 extra hosts are idle. An idle host still draws 25 W, and 2,493 x 25 W is
about 62 kW, on top of the 22 kW of the 277 hosts.
Overprovisioning could lead to sustainability concerns, both environmental and financial, where energy (and thus,
money) is wasted by idle infrastructure.
</details>

*Question:* what happens with 1/2 and 1/4 of the hosts?

<details><summary>Answer</summary>

With 1/2 of the hosts (138), the power draw drops by only 16%: switching off 139 idle hosts saves 139 x 25 W,
about 3.5 kW, while the jobs need the same energy as before.
At peak, up to 160 jobs run at the same time, more than 138 hosts, so some jobs wait for a free host.
With 1/4 of the hosts (69), jobs wait on average more than a day, and hundreds of jobs are still waiting at the end of
the day; the power draw is lower because these jobs did not run yet.
Under-provisioning ICT infrastructure could lead to performance concerns, system faults, and, ultimately, system
failures.
</details>

In [ ]:
plot_power(*runs.values());  # optional: the four simulations in one plot

## 3. Hosts Needed over the Day

Every SURF job uses one whole host (16 cores). Therefore, the number of hosts in use equals the number of running jobs,
which OpenDT predicts at every moment.
A dynamic autoscaler keeps these hosts on, plus a few spare hosts for new jobs, and switches off the rest.

*Question:* how many of the 277 hosts does SURF need during the day? Write down a guess, then run the next cell.

In [ ]:
SPARE = 10  # hosts kept on for new jobs

in_use = day1.service["tasks_active"]  # busy hosts at each moment (one job per host)
needed = (in_use + SPARE).clip(upper=277)

ax = in_use.plot(figsize=(10, 4), label="Hosts in use")
ax.fill_between(in_use.index, in_use, 277, alpha=0.2, label="Idle hosts")
ax.set(ylim=(0, 300), xlabel="Time [UTC]", ylabel="Hosts")
legend_above(ax)

In [ ]:
IDLE_POWER = 25  # watts an idle SURF host draws

switched_off = 277 - needed
saved_kwh = (switched_off * IDLE_POWER * 150).sum() / 3.6e6  # one sample per 150 s; joules to kWh
used_kwh = day1.summary()["energy_kwh"]
print(f"Switching off idle hosts saves {saved_kwh:.0f} of {used_kwh:.0f} kWh ({saved_kwh / used_kwh:.0%}).")

*Question:* what does this estimate ignore? What could go wrong when SURF follows this advice?

<details><summary>Answer</summary>

The estimate ignores that hosts need time (minutes) to boot.
When more jobs arrive than there are spare hosts, the new jobs wait until a host boots; this is why we keep spare
hosts.
A human-in-the-loop, aiming to ensure the correct and ethical operation of the process, oversees at least the major
decisions and can intervene at any time.
</details>

*Exercise 1:* set `SPARE = 0`, then `SPARE = 50`, and run the two cells above again. How much energy does SURF save in
each case? What could go wrong with `SPARE = 0`?

<details><summary>Answer</summary>

About 18% with no spare hosts, and about 12% with 50 spare hosts.
With no spare hosts, every new job waits until a host boots.
</details>

*Exercise 2 (optional):* what if each host draws only 10 W when idle, instead of 25 W? Run the next cell, then compare
the power draw and the MAPE with those of `day1`.

In [ ]:
low_idle = simulate(until=DAY, idle_power=10, label="OpenDT, 10 W idle")
plot_power(day1, low_idle)
print(f"Average power draw: {day1.summary()['avg_power_kw']:.1f} vs. {low_idle.summary()['avg_power_kw']:.1f} kW")
print(f"MAPE: {mape(day1):.1f}% vs. {mape(low_idle):.1f}%")

<details><summary>Answer</summary>

The average power draw drops from 21.7 kW to 19.0 kW (12% less).
However, the MAPE increases to about 18%: we changed the hardware, so the ground truth from SURF no longer describes
this datacenter.
</details>

## Bonus: Changing the Hosts in the Live Demo

*Optional.* In the live demo, we change the hosts from the REST API (`PUT /api/topology`). From that moment, OpenDT
predicts the datacenter with the new number of hosts.
We rebuild this view: until 08:00, the simulation with 277 hosts, and from 08:00, the simulation with 2,770 hosts.

In [ ]:
SWITCH = TRACE_START + pd.Timedelta("10h")  # 08:00 UTC on 7 October

before, after = day1.power["power_draw"][:SWITCH], runs[2770].power["power_draw"][SWITCH:]
scaled = pd.concat([before, after]) / 1000

ax = (measured_power(end=DAY) / 1000).plot(figsize=(10, 4), color="#999999", label="Ground Truth")
scaled.plot(ax=ax, label="OpenDT, 2,770 hosts from 08:00")
ax.set(ylim=(0, None), xlabel="Time [UTC]", ylabel="Power Draw [kW]")
legend_above(ax)

## Summary

In this notebook, we (1) validate OpenDT against the power draw measured at SURF, (2) explore what-if scenarios with
fewer and more hosts, and (3) identify how many hosts SURF needs over the day, as input for dynamic autoscaling.

*Next:* use case 2, simulation calibration, in [2.calibration.ipynb](2.calibration.ipynb).

*Learn more:* the [OpenDT documentation](https://atlarge-research.github.io/opendt/docs/), its [architecture](https://atlarge-research.github.io/opendt/docs/concepts/architecture) and
[topology configuration](https://atlarge-research.github.io/opendt/docs/configuration/topology), and the
[OpenDT paper](https://atlarge-research.com/pdfs/2026-icpe-opendt.pdf).